In [1]:
import json
import os

BASE = r"C:\Users\Win10\OneDrive - Università degli Studi di Torino\Desktop\repo_dss\dss_lab_project"
CORRECT_IDS_DIR = os.path.join(BASE, "dataset", "correct_ids")

print(f"\nAnalizzo i file dentro: {CORRECT_IDS_DIR}\n")

for filename in os.listdir(CORRECT_IDS_DIR):
    if not filename.endswith(".json"):
        continue

    path = os.path.join(CORRECT_IDS_DIR, filename)
    print("=" * 80)
    print(f"FILE: {filename}")

    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception as e:
        print(f"Errore nel leggere {filename}: {e}")
        continue

    # Non sappiamo se è lista o dict → gestiamo entrambi
    if isinstance(data, list):
        print(f"→ Numero righe: {len(data)}")
        if len(data) > 0:
            first = data[0]
            print(f"→ Colonne: {list(first.keys())}")

            # tipi
            types = {k: type(v).__name__ for k, v in first.items()}
            print(f"→ Tipi dei valori: {types}")

            # preview 3 righe
            print("\nPrime 3 righe:")
            for row in data[:3]:
                print(row)

    elif isinstance(data, dict):
        print("→ File è un dizionario, non una lista.")
        print(f"→ Chiavi principali: {list(data.keys())}")
        print("→ Primo elemento se possibile:")
        first_key = next(iter(data))
        print(data[first_key])

    print("\n")



Analizzo i file dentro: C:\Users\Win10\OneDrive - Università degli Studi di Torino\Desktop\repo_dss\dss_lab_project\dataset\correct_ids

FILE: artists.json
→ Numero righe: 104
→ Colonne: ['id_author', 'name', 'gender', 'birth_date', 'birth_place', 'nationality', 'description', 'active_start', 'active_end', 'province', 'region', 'country', 'latitude', 'longitude', 'type', 'active-end', 'new_id_artist']
→ Tipi dei valori: {'id_author': 'str', 'name': 'str', 'gender': 'str', 'birth_date': 'NoneType', 'birth_place': 'str', 'nationality': 'NoneType', 'description': 'str', 'active_start': 'str', 'active_end': 'NoneType', 'province': 'str', 'region': 'str', 'country': 'str', 'latitude': 'str', 'longitude': 'str', 'type': 'str', 'active-end': 'str', 'new_id_artist': 'str'}

Prime 3 righe:
{'id_author': 'ART82291002', 'name': '99 posse', 'gender': 'M', 'birth_date': None, 'birth_place': 'Napoli', 'nationality': None, 'description': 'gruppo musicale italiano', 'active_start': '1991-10-09', 'act

In [6]:
import json

TRACKS_PATH = r"C:\Users\Win10\OneDrive - Università degli Studi di Torino\Desktop\repo_dss\dss_lab_project\dataset\Finali\tracksFinal.json"

with open(TRACKS_PATH, "r", encoding="utf-8") as f:
    tracks = json.load(f)

print("Numero tracce:", len(tracks))

# Prendo la prima riga
first = tracks[0]

print("\nColonne presenti in tracksFinal.json:")
for col in sorted(first.keys()):
    print(" -", col)


Numero tracce: 11166

Colonne presenti in tracksFinal.json:
 - album
 - album_name
 - album_release_date
 - album_type
 - avg_token_per_clause
 - bpm
 - char_per_tok
 - date_id
 - day
 - disc_number
 - duration_ms
 - explicit
 - featured_artists
 - flatness
 - flux
 - id
 - id_album
 - id_artist
 - language
 - loudness
 - lyrics
 - month
 - n_sentences
 - n_tokens
 - new_id_album
 - new_id_artist
 - new_track_id
 - pitch
 - popularity
 - primary_artist
 - rms
 - rolloff
 - spectral_complexity
 - streams@1month
 - swear_EN
 - swear_EN_words
 - swear_IT
 - swear_IT_words
 - title
 - track_number
 - year


In [7]:
import json
import os
import math
from uuid import uuid4
from collections import defaultdict

# ============================================================
# 1) PATH
# ============================================================

BASE = r"C:\Users\Win10\OneDrive - Università degli Studi di Torino\Desktop\repo_dss\dss_lab_project\dataset"

IN_CORRECT = os.path.join(BASE, "correct_ids")

artists_in = os.path.join(IN_CORRECT, "artists.json")
tracks_in  = os.path.join(IN_CORRECT, "tracks.json")
parts_in   = os.path.join(IN_CORRECT, "participations.json")

OUT = os.path.join(BASE, "Finali")
os.makedirs(OUT, exist_ok=True)

artists_out = os.path.join(OUT, "artistsFinal.json")
tracks_out  = os.path.join(OUT, "tracksFinal.json")
parts_out   = os.path.join(OUT, "participationsFinal.json")

# ============================================================
# 2) FUNZIONI PER PULIZIA
# ============================================================

def is_raw_missing(v):
    if v is None:
        return True
    if isinstance(v, float) and math.isnan(v):
        return True
    if isinstance(v, str) and v.strip().lower() in ["", "null", "none", "nan", "undefined"]:
        return True
    return False

def normalize_missing(v):
    return "NULL" if is_raw_missing(v) else v

def to_float(v):
    if v == "NULL":
        return "NULL"
    try:
        return float(v)
    except:
        return "NULL"

def to_int(v):
    if v == "NULL":
        return "NULL"
    try:
        return int(float(v))
    except:
        return "NULL"

def to_bool(v):
    if v == "NULL":
        return "NULL"
    if isinstance(v, bool):
        return v
    if isinstance(v, str):
        v = v.lower()
        if v in ["true", "1", "yes"]:
            return True
        if v in ["false", "0", "no"]:
            return False
    return "NULL"

# ============================================================
# 3) LOAD FILES
# ============================================================

with open(artists_in, "r", encoding="utf-8") as f:
    artists = json.load(f)

with open(tracks_in, "r", encoding="utf-8") as f:
    tracks = json.load(f)

with open(parts_in, "r", encoding="utf-8") as f:
    parts = json.load(f)

print(f"Loaded {len(artists)} artists, {len(tracks)} tracks, {len(parts)} participations.")


# ============================================================
# 4) CLEAN ARTISTS
# ============================================================

for a in artists:

    # normalize
    for col, val in list(a.items()):
        a[col] = normalize_missing(val)

    # guarantee type field
    if "type" not in a or is_raw_missing(a["type"]):
        a["type"] = "NULL"
    else:
        a["type"] = normalize_missing(a["type"])

    # remove useless column
    if "active-end" in a:
        del a["active-end"]

    # convert coordinates
    a["latitude"]  = to_float(a.get("latitude", "NULL"))
    a["longitude"] = to_float(a.get("longitude", "NULL"))

    # gender uppercase
    if a["gender"] != "NULL":
        a["gender"] = a["gender"].upper()

print("✔ Artists cleaned.")


# ============================================================
# 5) CLEAN TRACKS + CREATE LYRICS/SYMPHONY IDS
# ============================================================

float_cols = [
    "n_sentences","n_tokens","char_per_tok","avg_token_per_clause",
    "bpm","rolloff","flux","rms","flatness","spectral_complexity",
    "pitch","loudness"
]

int_cols = [
    "year","month","day","disc_number","track_number","duration_ms","popularity"
]

for t in tracks:

    # normalize missing
    for col, val in list(t.items()):
        t[col] = normalize_missing(val)

    # convert ints
    for col in int_cols:
        t[col] = to_int(t.get(col, "NULL"))

    # convert floats
    for col in float_cols:
        t[col] = to_float(t.get(col, "NULL"))

    # explicit → bool/NULL
    t["explicit"] = to_bool(t.get("explicit", "NULL"))

    # ===================================================
    # CREATE NEW IDs IF NOT PRESENT
    # ===================================================
    if "lyrics_id" not in t or t["lyrics_id"] == "NULL":
        t["lyrics_id"] = "LYR_" + str(uuid4())

    if "symph_id" not in t or t["symph_id"] == "NULL":
        t["symph_id"] = "SYN_" + str(uuid4())

print("✔ Tracks cleaned + Lyrics/Symphony IDs generated.")


# ============================================================
# 6) CLEAN PARTICIPATION
# ============================================================

for p in parts:

    for col, val in list(p.items()):
        p[col] = normalize_missing(val)

    if "isPrimary" in p:
        p["IsPrimary"] = int(p["isPrimary"])
        del p["isPrimary"]
    elif "IsPrimary" in p:
        p["IsPrimary"] = int(p["IsPrimary"])
    else:
        p["IsPrimary"] = 0

print("✔ Participation cleaned.")


# ============================================================
# 7) CHECK MISSING
# ============================================================

def check_missing(data, name):
    print(f"\n===== CHECK MISSING IN {name} =====")

    all_ok = True
    all_cols = set()

    for row in data:
        all_cols.update(row.keys())

    for col in all_cols:
        for row in data:
            val = row.get(col, "NULL")
            if val == "NULL":
                continue
            if isinstance(val, str) and val.strip().lower() in ["", "nan", "none", "undefined"]:
                print(f"⚠️ Sporco residuo in {name}.{col}: {val}")
                all_ok = False

    if all_ok:
        print("✔ Tutti i missing sono 'NULL'.")


# ============================================================
# 8) CHECK MIXED TYPES
# ============================================================

def detect_mixed_types(data, name):
    print(f"\n===== CHECK TIPI MISTI: {name} =====")

    type_map = defaultdict(set)
    all_cols = set()

    for row in data:
        all_cols.update(row.keys())

    for row in data:
        for col in all_cols:
            val = row.get(col, "NULL")
            if val == "NULL":
                continue
            type_map[col].add(type(val).__name__)

    mixed = {col: t for col, t in type_map.items() if len(t) > 1}

    if mixed:
        print("⚠️ TIPI MISTI TROVATI:")
        for col, types in mixed.items():
            print(f" - {col}: {types}")
    else:
        print("✔ Nessun tipo misto (escludendo NULL).")


# ============================================================
# RUN CHECKS
# ============================================================

check_missing(artists, "ARTISTS")
check_missing(tracks, "TRACKS")
check_missing(parts,  "PARTICIPATION")

detect_mixed_types(artists, "ARTISTS")
detect_mixed_types(tracks,  "TRACKS")
detect_mixed_types(parts,   "PARTICIPATION")


# ============================================================
# 9) SAVE CLEAN DATASETS
# ============================================================

with open(artists_out, "w", encoding="utf-8") as f:
    json.dump(artists, f, indent=4, ensure_ascii=False)

with open(tracks_out, "w", encoding="utf-8") as f:
    json.dump(tracks, f, indent=4, ensure_ascii=False)

with open(parts_out, "w", encoding="utf-8") as f:
    json.dump(parts, f, indent=4, ensure_ascii=False)

print("\n=== FILE FINALI SALVATI ===")
print(" -", artists_out)
print(" -", tracks_out)
print(" -", parts_out)

Loaded 104 artists, 11166 tracks, 12622 participations.
✔ Artists cleaned.
✔ Tracks cleaned + Lyrics/Symphony IDs generated.
✔ Participation cleaned.

===== CHECK MISSING IN ARTISTS =====
✔ Tutti i missing sono 'NULL'.

===== CHECK MISSING IN TRACKS =====
✔ Tutti i missing sono 'NULL'.

===== CHECK MISSING IN PARTICIPATION =====
✔ Tutti i missing sono 'NULL'.

===== CHECK TIPI MISTI: ARTISTS =====
✔ Nessun tipo misto (escludendo NULL).

===== CHECK TIPI MISTI: TRACKS =====
✔ Nessun tipo misto (escludendo NULL).

===== CHECK TIPI MISTI: PARTICIPATION =====
✔ Nessun tipo misto (escludendo NULL).

=== FILE FINALI SALVATI ===
 - C:\Users\Win10\OneDrive - Università degli Studi di Torino\Desktop\repo_dss\dss_lab_project\dataset\Finali\artistsFinal.json
 - C:\Users\Win10\OneDrive - Università degli Studi di Torino\Desktop\repo_dss\dss_lab_project\dataset\Finali\tracksFinal.json
 - C:\Users\Win10\OneDrive - Università degli Studi di Torino\Desktop\repo_dss\dss_lab_project\dataset\Finali\parti

In [8]:
print(sorted(json.load(open(tracks_out))[0].keys()))


['album', 'album_name', 'album_release_date', 'album_type', 'avg_token_per_clause', 'bpm', 'char_per_tok', 'date_id', 'day', 'disc_number', 'duration_ms', 'explicit', 'featured_artists', 'flatness', 'flux', 'id', 'id_album', 'id_artist', 'language', 'loudness', 'lyrics', 'lyrics_id', 'month', 'n_sentences', 'n_tokens', 'new_id_album', 'new_id_artist', 'new_track_id', 'pitch', 'popularity', 'primary_artist', 'rms', 'rolloff', 'spectral_complexity', 'streams@1month', 'swear_EN', 'swear_EN_words', 'swear_IT', 'swear_IT_words', 'symph_id', 'title', 'track_number', 'year']
